# 08 — Sensitivity: normative tractogram streamline density

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from analysis.lib.data import load_label, restricted_pool, build_demographics_matrix
from analysis.lib.stacking import process_stacking_fold
from analysis.lib.cv import pooled_cv
from analysis.lib.stats import nadeau_bengio_test, bh_fdr

DATA_DIR        = PROJECT_ROOT / "data"
PROD_LACUNA_DIR = DATA_DIR / "lacuna" / "METAVCI_PSCI_LACUNA" / "derivatives"
SUB_BASE        = DATA_DIR / "lacuna" / "subsamples"
OUT_DIR         = PROJECT_ROOT / "outputs" / "predictive_modeling_sensitivity_streamlines"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLIN_PATH = DATA_DIR / "tabular" / "clinical_data.csv"
assert CLIN_PATH.exists(), f"missing {CLIN_PATH} — see data/README.md"

TARGETS = [
    "cognition-PSCI_present",
    "cognition-aef_domain_score",
    "cognition-ps_domain_score",
    "cognition-language_domain_score",
    "cognition-verbalmemory_domain_score",
    "cognition-vsfunctions_domain_score",
    "cognition-vsmemory_domain_score",
]
SUBSAMPLE_PCTS = [10, 25, 50, 75]


def _slnm_for_arm(arm):
    if arm == "full":
        deriv = PROD_LACUNA_DIR
    else:
        pct = int(arm.replace("sub", ""))
        deriv = SUB_BASE / f"METAVCI_PSCI_LACUNA_sub{pct}pct" / "derivatives"
    return load_label(deriv, "acuteinfarct", "snm", "disconnectionpct")

In [ ]:
clinical = pd.read_csv(CLIN_PATH, index_col=0)
pool = restricted_pool(PROD_LACUNA_DIR, clinical, ["acuteinfarct"])
clin_s = clinical.loc[pool]
print(f"acute pool: N = {len(pool)}")

lesion = load_label(PROD_LACUNA_DIR, "acuteinfarct", "rd",  "damagepct").reindex(pool)
flnm   = load_label(PROD_LACUNA_DIR, "acuteinfarct", "fnm", "tmap").reindex(pool)
X_demo, _, _, _, _ = build_demographics_matrix(clin_s, exclude_cohort=False)
n_demo = X_demo.shape[1]
cv = pooled_cv(seed=42)


def run_arm(arm, slnm):
    out_csv = OUT_DIR / f"stacking_scores_{arm}.csv"
    if out_csv.exists():
        print(f"[skip] {arm}: {out_csv.name} exists")
        return pd.read_csv(out_csv)
    print(f"=== arm: {arm} ===")
    arm_rows = []
    for target in TARGETS:
        y_series = clin_s[target]
        valid = y_series.notna()
        y = y_series[valid].values.astype(int)
        if y.sum() < 10 or len(y) - y.sum() < 10:
            print(f"  [skip] {target}")
            continue
        Xb = X_demo[valid.values]
        view_configs = [
            ("baseline",     Xb),
            ("lesion_acute", np.hstack([Xb, lesion.loc[pool].values[valid.values]])),
            ("flnm_acute",   np.hstack([Xb, flnm.loc[pool].values[valid.values]])),
            ("slnm_acute",   np.hstack([Xb, slnm.loc[pool].values[valid.values]])),
        ]
        fold_results = Parallel(n_jobs=-1)(
            delayed(process_stacking_fold)(i, tr, te, view_configs, y, n_demo)
            for i, (tr, te) in enumerate(cv.split(Xb, y)))
        for r in fold_results:
            r["Target"] = target
            r["Arm"] = arm
        arm_rows.extend(fold_results)
        print(f"  {target}: mean AUPRC = {np.nanmean([r['AUPRC'] for r in fold_results]):.4f}")
    df = pd.DataFrame(arm_rows)
    df.to_csv(out_csv, index=False)
    return df


per_arm = {}
for arm in ["full"] + [f"sub{p}" for p in SUBSAMPLE_PCTS]:
    slnm = _slnm_for_arm(arm).reindex(pool)
    per_arm[arm] = run_arm(arm, slnm)

In [ ]:
# NB significance per target: each subsample arm vs full (two-sided), BH-FDR within target
full = per_arm["full"]
rows = []
for arm in [f"sub{p}" for p in SUBSAMPLE_PCTS]:
    sub = per_arm[arm]
    for target in TARGETS:
        f = full[full["Target"] == target].set_index("Fold")["AUPRC"]
        s = sub[sub["Target"] == target].set_index("Fold")["AUPRC"]
        common = f.index.intersection(s.index)
        if len(common) < 2:
            continue
        res = nadeau_bengio_test(s.loc[common].values, f.loc[common].values,
                                 n_splits=10, test_size=0.1, alternative="two-sided")
        rows.append({"Arm": arm, "Target": target,
                     "mean_full": float(f.loc[common].mean()),
                     "mean_sub": float(s.loc[common].mean()),
                     "mean_diff": float(res["mean_diff"]),
                     "t_stat": float(res["t_stat"]), "p": float(res["p"]),
                     "n_paired": int(res["n_paired"])})
nb = pd.DataFrame(rows)
nb["p_adj_BH"] = nb.groupby("Target")["p"].transform(lambda x: bh_fdr(x.values))
nb.to_csv(OUT_DIR / "nb_significance.csv", index=False)
print(nb.round(4).to_string(index=False))

long = pd.concat(per_arm.values(), ignore_index=True)
summ = long.groupby(["Target", "Arm"])["AUPRC"].agg(["mean", "std", "count"]).reset_index()
summ["sem"] = summ["std"] / np.sqrt(summ["count"])
summ.to_csv(OUT_DIR / "summary.csv", index=False)
print(f"saved summaries to {OUT_DIR}")